# 🏠 Airbnb Price Oracle
### Eden Elfassy & Léonie Chapelle
Prédiction du logarithme du prix des logements Airbnb à partir de données brutes.

## Phase 1 — Exploration des données
### Étape 1 — Chargement & inspection des données

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuration de l'affichage
pd.set_option('display.max_columns', None)  # Voir toutes les colonnes
pd.set_option('display.max_colwidth', 50)   # Limiter la largeur des colonnes texte
plt.style.use('seaborn-v0_8')               # Style des graphiques

In [ ]:
# Chargement des données
train = pd.read_csv('../data/raw/airbnb_train.csv')
test = pd.read_csv('../data/raw/airbnb_test.csv')

print(f"Train : {train.shape[0]} lignes, {train.shape[1]} colonnes")
print(f"Test  : {test.shape[0]} lignes, {test.shape[1]} colonnes")

In [ ]:
train.head(3)

In [ ]:
# Aperçu du test set — ON vérifie qu'il Y a les mêmes colonnes (sauf log_price)
print("Colonnes présentes dans train mais pas dans test :")
print(set(train.columns) - set(test.columns))
print("\nColonnes présentes dans test mais pas dans train :")
print(set(test.columns) - set(train.columns))
test.head(3)

In [ ]:
# Vue d'ensemble : types de données et valeurs manquantes
missing = pd.DataFrame({
    'Type'     : train.dtypes,
    'Manquants': train.isnull().sum(),
    '% manquants': (train.isnull().sum() / len(train) * 100).round(1)
})

print(missing[missing['Manquants'] > 0].sort_values('% manquants', ascending=False))

In [ ]:
# Statistiques descriptives des variables numériques
# Permet de détecter : valeurs aberrantes (min/max extrêmes),
# asymétrie (différence médiane/moyenne), et l'amplitude de chaque variable
train.describe().round(2)

In [ ]:
# Vérification des doublons — un même logement ne doit apparaître qu'une fois
n_duplicates = train.duplicated(subset='id').sum()
print(f"Nombre de lignes dupliquées (même id) : {n_duplicates}")
# Si > 0, il faudrait les supprimer avec train.drop_duplicates(subset='id')

### 📋 Observations étape 1

- **22 234 lignes** dans le train set
- **4 colonnes avec beaucoup de valeurs manquantes** : 
  `host_response_rate` (24.6%), `review_scores_rating` (22.4%), 
  `first_review` (21.3%), `last_review` (21.2%)
- `host_response_rate` est stocké comme string ("100%") alors que c'est un nombre → à convertir
- `bathrooms`, `beds`, `bedrooms` ont moins de 0.2% de manquants → faciles à imputer
- `log_price` varie entre 2.30 et 7.60 → prix réels entre environs 10€ et 2000€
- `number_of_reviews` très asymétrique : médiane à 6 mais max à 505

### Étape 2 — Analyse des variables
On catégorise chaque colonne pour identifier comment la traiter dans les étapes suivantes.

In [ ]:
# Catégorisation des variables
numeriques = ['accommodates', 'bathrooms', 'bedrooms', 'beds', 
              'number_of_reviews', 'review_scores_rating',
              'latitude', 'longitude']

categorielles_nominales = ['room_type', 'property_type', 
                           'city', 'bed_type']

categorielles_ordinales = ['cancellation_policy']

booleennes = ['cleaning_fee', 'instant_bookable', 
              'host_has_profile_pic', 'host_identity_verified']

dates = ['first_review', 'last_review', 'host_since']

texte_libre = ['amenities', 'description', 'name', 'neighbourhood']

cible = ['log_price']

# Affichage propre
categories = {
    'Numériques'             : numeriques,
    'Catégorielles nominales': categorielles_nominales,
    'Catégorielles ordinales': categorielles_ordinales,
    'Booléennes'             : booleennes,
    'Dates'                  : dates,
    'Texte libre'            : texte_libre,
    'Cible'                  : cible
}

for cat, cols in categories.items():
    print(f"\n{cat} ({len(cols)}) :")
    for col in cols:
        print(f"   → {col}")

In [ ]:
# Vérification des valeurs uniques pour les variables catégorielles
for col in categorielles_nominales + categorielles_ordinales:
    print(f"\n{col} ({train[col].nunique()} valeurs uniques) :")
    print(train[col].value_counts().head(5))

In [ ]:
# Vérification bed_type
print(train['bed_type'].value_counts())

### 📋 Observations étape 2

- `room_type` : 3 valeurs bien équilibrées (Entire home: 12 348, Private room: 9 219, Shared room: 667) → OHE simple

- `property_type` : 31 valeurs uniques très déséquilibrées. Apartment domine avec 14 635 occurrences, 
  suivi de House (4 965). Mais beaucoup de types apparaissent moins de 50 fois (Yurt, Treehouse, Castle...). 
  Ces types ultra-rares n'apporteront aucune information utile au modèle car ils seront quasiment 
  toujours à 0. On les regroupera tous sous un label "Other" pour ne garder que les types vraiment représentatifs.

- `city` : 6 villes, NYC (9 739) et LA (6 672) très dominantes, Boston la moins représentée → OHE simple

- `cancellation_policy` : 5 valeurs dont super_strict_30 (37 occurrences) et super_strict_60 
  (5 occurrences) sont quasi absentes. On les regroupera avec "strict" lors de l'encodage ordinal 
  (flexible=0, moderate=1, strict=2). L'ordre a un sens naturel : plus c'est strict, plus c'est contraignant.

- `amenities` : liste d'équipements brute à parser spécifiquement en phase 2

- `description` et `name` : texte libre trop complexe et trop variable d'un logement à l'autre → ignorés

### Étape 3 — Nettoyage des données

In [ ]:
# On travaille sur une copie pour ne jamais modifier les données originales
train_clean = train.copy()
test_clean = test.copy()

In [ ]:
# Conversion des colonnes booléennes "t"/"f" en 0/1
bool_cols = ['cleaning_fee', 'instant_bookable', 
             'host_has_profile_pic', 'host_identity_verified']

for col in bool_cols:
    train_clean[col] = train_clean[col].map({'t': 1, 'f': 0})
    test_clean[col]  = test_clean[col].map({'t': 1, 'f': 0})

print("Booléennes converties ")
print(train_clean[bool_cols].head(3))

In [ ]:
# Conversion host_response_rate : "100%" → 100.0
def convert_response_rate(df):
    df['host_response_rate'] = df['host_response_rate'].str.replace('%', '').astype(float)
    median_rate = train_clean['host_response_rate'].median()
    df['host_response_rate'] = df['host_response_rate'].fillna(median_rate)
    return df

train_clean = convert_response_rate(train_clean)
test_clean  = convert_response_rate(test_clean)

print(f"host_response_rate converti   (médiane = {train_clean['host_response_rate'].median()}%)")

In [ ]:
# Création de has_review avant d'imputer
train_clean['has_review'] = train_clean['review_scores_rating'].notnull().astype(int)
test_clean['has_review']  = test_clean['review_scores_rating'].notnull().astype(int)

# Imputation par la médiane
median_rating = train_clean['review_scores_rating'].median()
train_clean['review_scores_rating'] = train_clean['review_scores_rating'].fillna(median_rating)
test_clean['review_scores_rating']  = test_clean['review_scores_rating'].fillna(median_rating)

print(f"has_review créé")
print(f"review_scores_rating imputé avec médiane = {median_rating}")

In [ ]:
# Imputation par la médiane groupée par room_type
for col in ['bathrooms', 'bedrooms', 'beds']:
    medians = train_clean.groupby('room_type')[col].median()
    
    for room in medians.index:
        mask_train = (train_clean['room_type'] == room) & (train_clean[col].isnull())
        mask_test  = (test_clean['room_type'] == room)  & (test_clean[col].isnull())
        
        train_clean.loc[mask_train, col] = medians[room]
        test_clean.loc[mask_test, col]   = medians[room]

print("bathrooms, bedrooms, beds imputés")
print(train_clean[['bathrooms', 'bedrooms', 'beds']].isnull().sum())

In [ ]:
# Vérification : plus aucune valeur manquante sur les colonnes traitées
cols_traitees = bool_cols + ['host_response_rate', 'review_scores_rating', 
                              'has_review', 'bathrooms', 'bedrooms', 'beds']

print("Valeurs manquantes après nettoyage :")
print(train_clean[cols_traitees].isnull().sum())

In [ ]:
train_clean['cleaning_fee'] = train_clean['cleaning_fee'].fillna(0)
test_clean['cleaning_fee']  = test_clean['cleaning_fee'].fillna(0)

In [ ]:
train_clean['host_has_profile_pic']    = train_clean['host_has_profile_pic'].fillna(0)
train_clean['host_identity_verified']  = train_clean['host_identity_verified'].fillna(0)
test_clean['host_has_profile_pic']     = test_clean['host_has_profile_pic'].fillna(0)
test_clean['host_identity_verified']   = test_clean['host_identity_verified'].fillna(0)

print("NaN restants corrigés")

### Étape 4 — Exploration visuelle des données (EDA)
L'objectif est de comprendre visuellement les données et de justifier 
nos choix de features pour la phase de modélisation.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogramme
axes[0].hist(train_clean['log_price'], bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Distribution du log_price')
axes[0].set_xlabel('log_price')
axes[0].set_ylabel('Nombre de logements')

# Boxplot global
axes[1].boxplot(train_clean['log_price'], vert=True, patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.7))
axes[1].set_title('Boxplot du log_price')
axes[1].set_ylabel('log_price')

plt.tight_layout()
plt.savefig('../outputs/distribution_log_price.png', dpi=150, bbox_inches='tight')
plt.show()

La distribution du `log_price` est approximativement normale, centrée autour de 4.78 
(soit ~120$ en valeur réelle). Quelques valeurs extrêmes visibles au-dessus de 7.0 
correspondent à des logements de luxe.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

cities = train_clean['city'].unique()
data_by_city = [train_clean[train_clean['city'] == city]['log_price'].values 
                for city in cities]

bp = ax.boxplot(data_by_city, labels=cities, patch_artist=True)

colors = ['steelblue', 'coral', 'mediumseagreen', 'mediumpurple', 'sandybrown', 'lightpink']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_title('Distribution du log_price par ville')
ax.set_xlabel('Ville')
ax.set_ylabel('log_price')

plt.tight_layout()
plt.savefig('../outputs/boxplot_by_city.png', dpi=150, bbox_inches='tight')
plt.show()

**SF affiche clairement les prix médians les plus élevés**, avec une boîte 
positionnée nettement au-dessus des autres villes. DC et Boston suivent de près, 
tandis que NYC — malgré sa réputation — affiche une médiane plus basse que SF, DC 
et Boston. LA et Chicago ont les prix médians les plus modérés.

On note également que LA a beaucoup plus d'outliers vers le haut (logements de luxe) 
que les autres villes, ce qui s'explique par la présence de villas et maisons haut 
de gamme typiques de Los Angeles.

La variable `city` est donc **très discriminante** pour la prédiction du prix 
et sera une feature essentielle dans nos modèles.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

room_types = train_clean['room_type'].unique()
data_by_room = [train_clean[train_clean['room_type'] == rt]['log_price'].values 
                for rt in room_types]

bp = ax.boxplot(data_by_room, labels=room_types, patch_artist=True)

colors = ['steelblue', 'coral', 'mediumseagreen']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_title('Distribution du log_price par type de logement')
ax.set_xlabel('Type de logement')
ax.set_ylabel('log_price')

plt.tight_layout()
plt.savefig('../outputs/boxplot_by_roomtype.png', dpi=150, bbox_inches='tight')
plt.show()

Le graphique confirme une hiérarchie très claire entre les 3 types de logements :

- `Entire home/apt` : médiane autour de 5.1 (~165$) : le plus cher logiquement 
  car on loue tout le logement
- `Private room` : médiane autour de 4.3 (~74$) : on partage les espaces communs
- `Shared room` : médiane autour de 3.9 (~49$) : le moins cher, on partage la chambre

Un point intéressant : `Private room` a énormément d'outliers vers le haut (jusqu'à 7.5) 
ce qui suggère que certaines chambres privées dans des logements de luxe peuvent 
atteindre des prix très élevés.

Cette hiérarchie très nette confirme que `room_type` sera l'une des variables 
les plus importantes de notre modèle.

In [ ]:
cols_corr = ['log_price', 'accommodates', 'bathrooms', 'bedrooms', 
             'beds', 'number_of_reviews', 'review_scores_rating',
             'host_response_rate', 'cleaning_fee', 'has_review']

corr_matrix = train_clean[cols_corr].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, ax=ax, square=True)
ax.set_title('Matrice de corrélation avec log_price')

plt.tight_layout()
plt.savefig('../outputs/correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

### Lecture de la matrice de corrélation

La première ligne/colonne `log_price` nous donne les corrélations 
avec le prix — c'est ce qui nous intéresse le plus :

**Corrélations positives fortes avec log_price :**
- `accommodates` : 0.56 : plus le logement accueille de personnes, plus il est cher
- `bedrooms` : 0.47 : plus il y a de chambres, plus c'est cher
- `bathrooms` : 0.37 : même logique
- `beds` : 0.44 : cohérent avec accommodates

Ces 4 variables mesurent toutes la **taille du logement** et sont 
fortement liées entre elles (ex: accommodates/beds = 0.82).

**Corrélations faibles ou nulles :**
- `number_of_reviews` : -0.04 : quasi nul, le nombre d'avis n'influence pas le prix
- `review_scores_rating` : 0.09 : la note non plus, surprenant mais cohérent
- `host_response_rate` : 0.01 : aucun lien avec le prix
- `cleaning_fee` : données manquantes, à investiguer

**Conclusion :** Les variables de taille du logement (`accommodates`, 
`bedrooms`, `bathrooms`, `beds`) sont les plus corrélées au prix parmi 
les variables numériques. Les variables liées aux avis et à l'hôte 
ont peu d'impact direct sur le prix.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

for rt, color in zip(['Entire home/apt', 'Private room', 'Shared room'], 
                     ['steelblue', 'coral', 'mediumseagreen']):
    subset = train_clean[train_clean['room_type'] == rt]
    ax.scatter(subset['bedrooms'], subset['log_price'], 
               alpha=0.3, color=color, label=rt, s=10)

ax.set_title('log_price en fonction du nombre de chambres')
ax.set_xlabel('Nombre de chambres (bedrooms)')
ax.set_ylabel('log_price')
ax.legend()

plt.tight_layout()
plt.savefig('../outputs/scatter_bedrooms.png', dpi=150, bbox_inches='tight')
plt.show()

### Lecture du scatter bedrooms vs log_price

On observe une tendance claire : **plus il y a de chambres, plus le prix monte**.

**Points clés :**
- La majorité des logements ont 0, 1 ou 2 chambres (colonnes très denses à gauche)
- Les logements `Entire home/apt` (bleu) dominent à partir de 2 chambres et 
  atteignent les prix les plus élevés
- Les `Private room` (orange) se concentrent sur 0-1 chambre avec des prix plus bas
- Les `Shared room` (vert) sont quasi exclusivement à 0-1 chambre, prix les plus bas
- Au-delà de 5 chambres les points se raréfient : ce sont des logements atypiques 
  (grandes villas, maisons familiales)

**Conclusion :** `bedrooms` est une feature utile mais son effet sur le prix 
est fortement lié au `room_type` : les deux variables interagissent ensemble, 
ce qu'on exploitera en phase 2 avec nos ratios.

In [ ]:
# On garde les 10 types les plus fréquents
top_types = train_clean['property_type'].value_counts().head(10)

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.bar(top_types.index, top_types.values, color='steelblue', alpha=0.8)

ax.set_title('Top 10 des types de logements')
ax.set_xlabel('Type de logement')
ax.set_ylabel('Nombre de logements')
plt.xticks(rotation=45, ha='right')

plt.tight_layout()
plt.savefig('../outputs/property_type_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

### Lecture du bar chart property_type

Le déséquilibre est frappant : `Apartment` domine massivement avec ~14 600 
logements, soit presque **3x plus** que le deuxième type (`House` ~5 000).

À partir de `Guesthouse`, les barres deviennent quasi invisibles : ces types 
ont moins de 200 occurrences chacun sur 22 000 logements.

**Conséquence pour la phase 2 :** On gardera uniquement les 5 types 
bien représentés (Apartment, House, Condominium, Townhouse, Loft) et on 
regroupera tout le reste en "Other". Cela évitera de créer des colonnes 
quasi vides qui pollueraient le modèle.

### 📋 Bilan Phase 1 — Exploration

Les variables les plus prometteuses identifiées lors de l'EDA :
- `city` : forte variation des prix selon la ville (SF > DC > Boston > NYC > Chicago > LA)
- `room_type` : hiérarchie très claire (Entire > Private > Shared)
- `accommodates`, `bedrooms`, `bathrooms`, `beds` : corrélés positivement au prix (taille du logement)
- `property_type` : à encoder après regroupement — garder top 5 + "Other"
- `cleaning_fee` : à investiguer en phase 2

Ces observations guideront nos choix de features en Phase 2.

## Phase 2 — Feature Engineering
### Étape 5 — Encodage des variables catégorielles

On transforme les variables textuelles en variables numériques 
utilisables par les algorithmes de machine learning.

In [ ]:
# On part de train_clean et test_clean de la phase 1
train_fe = train_clean.copy()
test_fe  = test_clean.copy()

In [ ]:
# Encodage ordinal de cancellation_policy
# flexible < moderate < strict < super_strict
cancellation_map = {
    'flexible'        : 0,
    'moderate'        : 1,
    'strict'          : 2,
    'super_strict_30' : 2,  # regroupé avec strict
    'super_strict_60' : 2   # regroupé avec strict
}

train_fe['cancellation_policy'] = train_fe['cancellation_policy'].map(cancellation_map)
test_fe['cancellation_policy']  = test_fe['cancellation_policy'].map(cancellation_map)

print("cancellation_policy encodé")
print(train_fe['cancellation_policy'].value_counts())

In [ ]:
# Regroupement des property_type rares en "Other"
top_property_types = train_fe['property_type'].value_counts()
top_property_types = top_property_types[top_property_types >= 50].index.tolist()

train_fe['property_type'] = train_fe['property_type'].apply(
    lambda x: x if x in top_property_types else 'Other'
)
test_fe['property_type'] = test_fe['property_type'].apply(
    lambda x: x if x in top_property_types else 'Other'
)

print("property_type nettoyé")
print(f"Nombre de types gardés : {train_fe['property_type'].nunique()}")
print(train_fe['property_type'].value_counts())

In [ ]:
# One Hot Encoding sur room_type, city, bed_type, property_type
cols_ohe = ['room_type', 'city', 'bed_type', 'property_type']

train_fe = pd.get_dummies(train_fe, columns=cols_ohe, prefix=cols_ohe)
test_fe  = pd.get_dummies(test_fe,  columns=cols_ohe, prefix=cols_ohe)

# Aligner les colonnes train/test (au cas où le test aurait des valeurs manquantes)
train_fe, test_fe = train_fe.align(test_fe, join='left', axis=1, fill_value=0)

print("OHE appliqué")
print(f"Nouvelles colonnes créées : {[c for c in train_fe.columns if any(p in c for p in cols_ohe)]}")

### 📋 Observations encodage

- `cancellation_policy` : encodé en ordinal 0/1/2 : l'ordre a un sens naturel
- `property_type` : types avec moins de 50 occurrences regroupés en "Other" 
  pour éviter des colonnes quasi vides
- `room_type`, `city`, `bed_type`, `property_type` : transformés en colonnes 
  binaires via One Hot Encoding
- `.align()` garantit que train et test ont exactement les mêmes colonnes 
  après OHE — indispensable pour que le modèle fonctionne correctement

### Étape 6 — Parsing des amenities

La colonne `amenities` contient une liste brute d'équipements sous forme de texte
ex: `{TV,"Wireless Internet",Kitchen,...}`

On va extraire les équipements les plus importants et créer des colonnes binaires 
(1 = équipement présent, 0 = absent) ainsi qu'un compteur total d'équipements.

In [ ]:
import re

def parse_amenities(amenities_str):
    """Convertit la string brute en liste propre d'équipements"""
    if pd.isna(amenities_str):
        return []
    # Supprime les accolades et les guillemets
    cleaned = re.sub(r'[{}""]', '', amenities_str)
    # Sépare par virgule et nettoie les espaces
    items = [item.strip().lower() for item in cleaned.split(',')]
    return [item for item in items if item]  # supprime les items vides

# Test sur un exemple
exemple = train_fe['amenities'].iloc[0]
print("Brut :", exemple[:100])
print("Parsé :", parse_amenities(exemple))

In [ ]:
# Dictionnaire : nom de la feature → mot clé à chercher dans les amenities
amenities_to_extract = {
    'has_wifi'        : 'wireless internet',
    'has_tv'          : 'tv',
    'has_kitchen'     : 'kitchen',
    'has_ac'          : 'air conditioning',
    'has_parking'     : 'parking',
    'has_washer'      : 'washer',
    'has_dryer'       : 'dryer',
    'has_gym'         : 'gym',
    'has_elevator'    : 'elevator',
    'has_doorman'     : 'doorman',
    'has_breakfast'   : 'breakfast',
    'has_pets_allowed': 'pets allowed'
}

def extract_amenity_features(df):
    # Nombre total d'équipements
    df['n_amenities'] = df['amenities'].apply(
        lambda x: len(parse_amenities(x))
    )
    
    # Colonnes binaires pour chaque équipement
    for feature_name, keyword in amenities_to_extract.items():
        df[feature_name] = df['amenities'].apply(
            lambda x: 1 if keyword in ' '.join(parse_amenities(x)) else 0
        )
    
    return df

train_fe = extract_amenity_features(train_fe)
test_fe  = extract_amenity_features(test_fe)

print("Amenities parsées")
print(f"\nNombre moyen d'équipements : {train_fe['n_amenities'].mean():.1f}")
print(f"\nTaux de présence de chaque équipement :")
amenity_cols = list(amenities_to_extract.keys())
print((train_fe[amenity_cols].mean() * 100).round(1).sort_values(ascending=False))

In [ ]:
# Corrélation de chaque amenity avec log_price
amenity_cols_all = amenity_cols + ['n_amenities']
corr_amenities = train_fe[amenity_cols_all + ['log_price']].corr()['log_price']
corr_amenities = corr_amenities.drop('log_price').sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['steelblue' if v > 0 else 'coral' for v in corr_amenities.values]
ax.barh(corr_amenities.index, corr_amenities.values, color=colors, alpha=0.8)
ax.set_title('Corrélation des amenities avec log_price')
ax.set_xlabel('Corrélation')
ax.axvline(x=0, color='black', linewidth=0.8)

plt.tight_layout()
plt.savefig('../outputs/amenities_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

print(corr_amenities)

### 📋 Observations parsing amenities

**Résultats surprenants et intéressants :**

- `has_tv` est l'amenity **la plus corrélée** avec le prix (0.27) : surprenant ! 
  Cela reflète probablement que les logements bien équipés avec TV sont 
  des appartements entiers de standing
- `n_amenities` est très corrélé (0.20) : plus un logement a d'équipements, 
  plus il est cher, ce qui traduit un effet de standing global
- `has_washer` (0.17) et `has_dryer` (0.14) sont bien corrélés : 
  équipements typiques des appartements entiers de qualité
- `has_elevator`, `has_gym`, `has_doorman` (~0.12) confirment l'effet 
  immeuble haut de gamme en ville
- `has_breakfast` est **négatif** (-0.05) : les B&B qui proposent le petit 
  déjeuner sont généralement moins chers que les appartements entiers
- `has_wifi` et `has_parking` sont faiblement corrélés (~0.04) car 
  trop répandus pour être discriminants

**Conclusion :** Le parsing des amenities nous a permis de créer 13 nouvelles 
features utiles, dont plusieurs avec des corrélations significatives avec le prix.

### Étape 7 — Features temporelles

On extrait de l'information utile à partir des colonnes de dates.
Une date brute n'est pas utilisable par un modèle — on la transforme 
en nombre de jours pour capturer des effets comme l'ancienneté de l'hôte.

In [ ]:
# Date de référence — on utilise la date max du dataset comme "aujourd'hui"
# la date de collecte des données du dataset (les dernières reviews datent de 2017).
# Utiliser une date fixe (plutôt que datetime.today()) garantit la reproductibilité :
# le notebook donnera toujours les mêmes résultats quel que soit le jour d'exécution.
reference_date = pd.Timestamp('2017-12-31')

def create_temporal_features(df):
    
    # Conversion des colonnes dates en datetime
    df['host_since']    = pd.to_datetime(df['host_since'],    errors='coerce')
    df['first_review']  = pd.to_datetime(df['first_review'],  errors='coerce')
    df['last_review']   = pd.to_datetime(df['last_review'],   errors='coerce')
    
    # Ancienneté de l'hôte en jours
    df['host_seniority_days'] = (reference_date - df['host_since']).dt.days
    df['host_seniority_days'] = df['host_seniority_days'].fillna(
        df['host_seniority_days'].median()
    )
    
    # Jours depuis le premier avis
    df['days_since_first_review'] = (reference_date - df['first_review']).dt.days
    df['days_since_first_review'] = df['days_since_first_review'].fillna(0)
    
    # Durée entre premier et dernier avis (activité du listing)
    df['review_span_days'] = (df['last_review'] - df['first_review']).dt.days
    df['review_span_days'] = df['review_span_days'].fillna(0)
    
    # Indicateur listing neuf (pas encore d'avis)
    df['has_no_reviews'] = (df['first_review'].isna()).astype(int)
    
    # Suppression des colonnes dates brutes — plus utiles
    df = df.drop(columns=['host_since', 'first_review', 'last_review'])
    
    return df

train_fe = create_temporal_features(train_fe)
test_fe  = create_temporal_features(test_fe)

print("Features temporelles créées")
print(f"\nhost_seniority_days  — moyenne : {train_fe['host_seniority_days'].mean():.0f} jours")
print(f"days_since_first_review — moyenne : {train_fe['days_since_first_review'].mean():.0f} jours")
print(f"review_span_days     — moyenne : {train_fe['review_span_days'].mean():.0f} jours")
print(f"has_no_reviews       — % listings neufs : {train_fe['has_no_reviews'].mean()*100:.1f}%")

In [ ]:
temp_cols = ['host_seniority_days', 'days_since_first_review', 
             'review_span_days', 'has_no_reviews']

corr_temp = train_fe[temp_cols + ['log_price']].corr()['log_price'].drop('log_price')
print("Corrélation des features temporelles avec log_price :")
print(corr_temp.sort_values(ascending=False).round(3))

### 📋 Observations features temporelles

**Statistiques :**
- En moyenne les hôtes sont sur la plateforme depuis **1 258 jours** (~3.5 ans)
- En moyenne le premier avis date d'il y a **564 jours** (~1.5 ans)
- La durée moyenne entre premier et dernier avis est de **335 jours**
- **21.3%** des listings n'ont aucun avis : ce sont des logements récents

**Corrélations avec log_price :**
- `has_no_reviews` : 0.096 : surprenant ! Les listings sans avis 
  sont légèrement plus chers, probablement des logements neufs 
  et récemment rénovés à prix premium
- `host_seniority_days` : 0.072 : les hôtes expérimentés tendent 
  à mieux tarifer leurs logements
- `review_span_days` : 0.018 : faible impact
- `days_since_first_review` : 0.001 : quasi nul, peu informatif

**Conclusion :** Les features temporelles ont des corrélations modestes 
mais restent utiles, notamment `has_no_reviews` et `host_seniority_days` 
qui capturent des effets réels sur le prix.

### Étape 8 — Features géographiques

On calcule la distance de chaque logement au centre-ville de sa ville.
Plus un logement est central, plus il tend à être cher.

On utilise la distance euclidienne (différence de coordonnées GPS) plutôt que la formule de Haversine (vraie distance en km) car :

→ Les distances intra-urbaines sont petites (< 50 km), 

→ l'approximation euclidienne est donc suffisamment précise

→ La formule est plus simple et plus rapide à calculer

→ Le modèle n'a pas besoin de la vraie distance en km

→ Juste d'un score relatif de "centralité"

In [ ]:
# Centres-villes approximatifs par ville
city_centers = {
    'NYC'    : (40.75, -73.99),
    'LA'     : (34.05, -118.24),
    'SF'     : (37.77, -122.42),
    'DC'     : (38.89, -77.03),
    'Chicago': (41.88, -87.63),
    'Boston' : (42.36, -71.06)
}

def calculate_distance(lat, lon, city, city_centers):
    """Calcule la distance euclidienne au centre-ville"""
    if city not in city_centers:
        return np.nan
    center_lat, center_lon = city_centers[city]
    return np.sqrt((lat - center_lat)**2 + (lon - center_lon)**2)

def create_geo_features(df):
    # Distance au centre-ville
    df['dist_center'] = df.apply(
        lambda row: calculate_distance(
            row['latitude'], row['longitude'], 
            row['city'], city_centers
        ), axis=1
    )
    df['dist_center'] = df['dist_center'].fillna(df['dist_center'].median())
    
    return df

# Attention : on doit récupérer la colonne city AVANT le OHE
# On la recrée temporairement à partir des colonnes OHE
def get_city_from_ohe(df):
    city_cols = [c for c in df.columns if c.startswith('city_')]
    cities = []
    for _, row in df[city_cols].iterrows():
        city = None
        for col in city_cols:
            if row[col] == 1:
                city = col.replace('city_', '')
                break
        cities.append(city)
    return cities

train_fe['city_temp'] = get_city_from_ohe(train_fe)
test_fe['city_temp']  = get_city_from_ohe(test_fe)

# Calcul de la distance
train_fe['dist_center'] = train_fe.apply(
    lambda row: calculate_distance(
        row['latitude'], row['longitude'],
        row['city_temp'], city_centers
    ), axis=1
)
test_fe['dist_center'] = test_fe.apply(
    lambda row: calculate_distance(
        row['latitude'], row['longitude'],
        row['city_temp'], city_centers
    ), axis=1
)

# Suppression de la colonne temporaire
train_fe = train_fe.drop(columns=['city_temp'])
test_fe  = test_fe.drop(columns=['city_temp'])

print("Features géographiques créées")
print(f"dist_center — moyenne : {train_fe['dist_center'].mean():.4f}")
print(f"dist_center — min : {train_fe['dist_center'].min():.4f}")
print(f"dist_center — max : {train_fe['dist_center'].max():.4f}")

In [ ]:
# Corrélation avec log_price
corr_dist = train_fe[['dist_center', 'log_price']].corr()['log_price']['dist_center']
print(f"Corrélation dist_center / log_price : {corr_dist:.3f}")

# Scatter dist_center vs log_price
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(train_fe['dist_center'], train_fe['log_price'], 
           alpha=0.1, color='steelblue', s=5)
ax.set_title('log_price en fonction de la distance au centre-ville')
ax.set_xlabel('Distance au centre-ville')
ax.set_ylabel('log_price')

plt.tight_layout()
plt.savefig('../outputs/scatter_dist_center.png', dpi=150, bbox_inches='tight')
plt.show()

### 📋 Observations features géographiques

- `dist_center` varie entre **0.0006** (très central) et **0.7163** (très excentré)
  avec une moyenne de **0.094** : la majorité des logements sont proches du centre
- Corrélation de **-0.137** avec log_price : corrélation négative confirmée :
  plus un logement est loin du centre-ville, moins il est cher
- Le scatter plot montre clairement que les prix élevés (>6.0) se concentrent 
  quasi exclusivement dans les 0.1 premiers degrés de distance
- Au-delà de 0.3 de distance, les prix chutent significativement et 
  les logements se raréfient

**Conclusion :** `dist_center` est une feature utile avec une corrélation 
de -0.137, ce qui en fait une des features géographiques les plus 
informatives du dataset. Les logements centraux sont systématiquement 
plus chers, ce qui est cohérent avec la réalité du marché immobilier urbain.

### Étape 9 — Ratios et interactions

On crée de nouvelles features en combinant des variables existantes.
Ces ratios capturent la **densité** du logement — informations que 
les variables brutes ne peuvent pas exprimer seules.

In [ ]:
def create_ratio_features(df):
    
    # Ratio lits / chambres — densité de couchage
    # +1 pour éviter la division par zéro
    df['beds_per_bedroom'] = df['beds'] / (df['bedrooms'] + 1)
    
    # Ratio personnes / chambres — densité d'occupation
    df['accommodates_per_bedroom'] = df['accommodates'] / (df['bedrooms'] + 1)
    
    # Ratio salles de bain / chambres — confort du logement
    df['bathrooms_per_bedroom'] = df['bathrooms'] / (df['bedrooms'] + 1)
    
    # Ratio personnes / lits — entassement
    df['accommodates_per_bed'] = df['accommodates'] / (df['beds'] + 1)
    
    return df

train_fe = create_ratio_features(train_fe)
test_fe  = create_ratio_features(test_fe)

print("Ratios créés")

# Corrélation avec log_price
ratio_cols = ['beds_per_bedroom', 'accommodates_per_bedroom', 
              'bathrooms_per_bedroom', 'accommodates_per_bed']

corr_ratios = train_fe[ratio_cols + ['log_price']].corr()['log_price'].drop('log_price')
print("\nCorrélation des ratios avec log_price :")
print(corr_ratios.sort_values(ascending=False).round(3))

### 📋 Observations ratios

**Corrélations avec log_price :**

- `accommodates_per_bed` : **0.439** : la corrélation la plus forte ! 
  Un logement avec beaucoup de personnes par lit est souvent un grand 
  appartement entier bien équipé, donc plus cher
- `accommodates_per_bedroom` : **0.342** : plus il y a de personnes 
  par chambre, plus le logement est grand et cher
- `beds_per_bedroom` : **0.196** : corrélation modérée, 
  les logements avec plusieurs lits par chambre tendent à être plus chers
- `bathrooms_per_bedroom` : **-0.023** : quasi nul, 
  ce ratio n'apporte pas vraiment d'information supplémentaire

**Conclusion :** Les ratios `accommodates_per_bed` et 
`accommodates_per_bedroom` sont de très bonnes features avec des 
corrélations de 0.439 et 0.342 : bien meilleures que les variables 
brutes `beds` (0.44) et `bedrooms` (0.47) prises séparément. 
Le feature engineering a clairement apporté de la valeur !

### Étape 10 — Nettoyage du neighbourhood

`neighbourhood` a trop de valeurs uniques pour un OHE direct.
On utilise le **target encoding** : on remplace chaque quartier 
par le prix moyen des logements de ce quartier calculé sur le train set.

C'est plus informatif qu'un OHE car ça capture directement 
l'effet du quartier sur le prix.

In [ ]:
# Target encoding du neighbourhood
# On calcule le prix moyen par quartier sur le train set uniquement

# Prix moyen global — sert de valeur par défaut pour les quartiers inconnus
global_mean = train_fe['log_price'].mean()

# Prix moyen par quartier sur le train
neighbourhood_mean = train_fe.groupby(
    train_clean['neighbourhood']
)['log_price'].mean()

print(f"Nombre de quartiers uniques : {len(neighbourhood_mean)}")
print(f"Prix moyen global : {global_mean:.3f}")
print(f"\nTop 10 quartiers les plus chers :")
print(neighbourhood_mean.sort_values(ascending=False).head(10).round(3))

In [ ]:
# neighbourhood_mean_price sera calculé proprement après le split
# pour éviter le data leakage
train_fe['neighbourhood_mean_price'] = 0
test_fe['neighbourhood_mean_price']  = 0

print("neighbourhood_mean_price initialisé à 0 — sera recalculé après split")

In [ ]:
# On n'a plus besoin de la colonne neighbourhood brute
# Elle a déjà été encodée en neighbourhood_mean_price
# On la supprime de train_clean et test_clean 
# pour ne pas la retrouver accidentellement plus tard

# Vérification qu'elle n'est plus dans train_fe/test_fe
if 'neighbourhood' in train_fe.columns:
    train_fe = train_fe.drop(columns=['neighbourhood'])
if 'neighbourhood' in test_fe.columns:
    test_fe  = test_fe.drop(columns=['neighbourhood'])

print("Colonne neighbourhood brute supprimée")
print(f"\nNombre total de colonnes dans train_fe : {train_fe.shape[1]}")

### 📋 Observations neighbourhood

- `neighbourhood_mean_price` est la **feature la plus corrélée au prix** 
  avec 0.492 — le quartier est un facteur déterminant du prix d'un logement
- Target encoding utilisé plutôt que OHE car `neighbourhood` a trop 
  de valeurs uniques
- Les quartiers inconnus dans le test set reçoivent le prix moyen 
  global comme valeur par défaut
- On calcule la moyenne uniquement sur le train set pour éviter le data leakage

### 🏁 Bilan Phase 2 — Feature Engineering

On est passés de **28 colonnes brutes** à **68 features numériques** 
prêtes pour la modélisation.

## Phase 3 — Modélisation
### Étape 11 — Split des données & modèle baseline

On sépare les données en train/validation (80/20) pour évaluer 
nos modèles. La métrique principale est le **RMSE** (Root Mean Square Error) 
sur le `log_price` — plus il est bas, meilleur est le modèle.

On commence par une régression linéaire simple comme **baseline** — 
tous nos modèles devront faire mieux que ce score de référence.

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import numpy as np

# Colonnes à exclure — non numériques ou inutiles pour le modèle
cols_to_drop = ['id', 'log_price', 'description', 'name', 
                'amenities', 'zipcode']

# On ne garde que les colonnes qui existent vraiment
cols_to_drop = [c for c in cols_to_drop if c in train_fe.columns]

# Séparation X et y
X = train_fe.drop(columns=cols_to_drop)
y = train_fe['log_price']

print(f"Shape de X : {X.shape}")
print(f"Shape de y : {y.shape}")
print(f"\nColonnes utilisées ({len(X.columns)}) :")
print(list(X.columns))

In [ ]:
# Split 80/20
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Train      : {X_train.shape[0]} lignes")
print(f"Validation : {X_val.shape[0]} lignes")

In [ ]:
# Recalcul propre de neighbourhood_mean_price APRÈS le split
# On utilise uniquement X_train pour éviter le data leakage

global_mean = y_train.mean()

neighbourhood_mean_clean = train_clean.loc[
    X_train.index, 'neighbourhood'
].to_frame().join(y_train).groupby('neighbourhood')['log_price'].mean()

# Application sur train et validation
train_fe['neighbourhood_mean_price'] = train_clean['neighbourhood'].map(
    neighbourhood_mean_clean
).fillna(global_mean)

# Application sur test
test_fe['neighbourhood_mean_price'] = test_clean['neighbourhood'].map(
    neighbourhood_mean_clean
).fillna(global_mean)

# Mise à jour de X et X_test avec la vraie valeur
X = train_fe.drop(columns=[c for c in ['id', 'log_price', 'description', 
                'name', 'amenities', 'zipcode'] if c in train_fe.columns])
X_test = test_fe.drop(columns=[c for c in ['id', 'description', 
                'name', 'amenities', 'zipcode'] if c in test_fe.columns])
X_test = X_test.reindex(columns=X.columns, fill_value=0)

# Nouveau split avec les données corrigées
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Data leakage corrigé")
print(f"Quartiers encodés : {len(neighbourhood_mean_clean)}")
print(f"Exemple valeurs : {train_fe['neighbourhood_mean_price'].describe().round(3)}")

In [ ]:
# Entraînement baseline
baseline = LinearRegression()
baseline.fit(X_train, y_train)

# Prédictions
y_pred_baseline = baseline.predict(X_val)

# RMSE
rmse_baseline = np.sqrt(mean_squared_error(y_val, y_pred_baseline))
print(f"Baseline Linear Regression — RMSE : {rmse_baseline:.4f}")

# On stocke les résultats dans un dictionnaire pour comparer plus tard
results = {}
results['Linear Regression'] = rmse_baseline

### 📋 Observations baseline

**Configuration :**
- **62 features** utilisées pour l'entraînement
- **17 787 lignes** pour entraîner le modèle, **4 447 lignes** pour valider
- Split 80/20 avec random_state=42 pour la reproductibilité

**Résultat baseline — Linear Regression : RMSE = 0.4125**

Le RMSE est notre métrique principale — plus il est bas, meilleur est le modèle.
Un RMSE de 0.4125 sur le log_price signifie qu'en moyenne notre modèle 
se trompe de 0.41 en valeur logarithmique, ce qui correspond à une erreur 
d'environ **51% sur le prix réel** (e^0.41 ≈ 1.51).

**Ce score est notre référence absolue** — tous les modèles testés 
ensuite devront obtenir un RMSE inférieur à **0.4125** pour être 
considérés comme meilleurs que cette baseline.

C'est un score correct pour une régression linéaire simple, mais 
on s'attend à faire bien mieux avec des modèles plus puissants 
comme Random Forest ou XGBoost qui peuvent capturer des relations 
non-linéaires entre les features.

### Étape 12 — Comparaison de 7 modèles

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.model_selection import cross_val_score
import time

In [ ]:
# Dictionnaire de tous les modèles à tester
models = {
    'Linear Regression'   : LinearRegression(),
    'Ridge'               : Ridge(alpha=1.0),
    'Lasso'               : Lasso(alpha=0.001),
    'Random Forest'       : RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'Gradient Boosting'   : GradientBoostingRegressor(n_estimators=100, random_state=42),
    'XGBoost'             : XGBRegressor(n_estimators=100, random_state=42, 
                                         verbosity=0, n_jobs=-1),
    'LightGBM'            : LGBMRegressor(n_estimators=100, random_state=42, 
                                          verbose=-1, n_jobs=-1)
}

In [ ]:
# Entraînement et évaluation de chaque modèle
print("Entraînement en cours...\n")
print(f"{'Modèle':<25} {'RMSE moyen':>12} {'Écart-type':>12} {'Temps':>8}")
print("-" * 62)

for name, model in models.items():
    start = time.time()
    
    # Cross-validation 5 folds — scoring négatif donc on prend -scores
    scores = cross_val_score(
        model, X, y, 
        cv=5, 
        scoring='neg_root_mean_squared_error',
        n_jobs=-1
    )
    rmse_mean = -scores.mean()
    rmse_std  = scores.std()
    elapsed   = time.time() - start
    
    # Stockage des résultats
    results[name] = rmse_mean
    
    print(f"{name:<25} {rmse_mean:>12.4f} {rmse_std:>12.4f} {elapsed:>7.1f}s")

print("\nTerminé")

In [ ]:
# Tri des modèles par RMSE croissant
results_sorted = dict(sorted(results.items(), key=lambda x: x[1]))

fig, ax = plt.subplots(figsize=(12, 6))
colors = ['coral' if v == min(results_sorted.values()) 
          else 'steelblue' for v in results_sorted.values()]

bars = ax.barh(list(results_sorted.keys()), 
               list(results_sorted.values()),
               color=colors, alpha=0.8)

# Ligne baseline
ax.axvline(x=results['Linear Regression'], 
           color='red', linestyle='--', 
           linewidth=1.5, label='Baseline')

# Valeurs sur les barres
for bar, val in zip(bars, results_sorted.values()):
    ax.text(val + 0.002, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=10)

ax.set_title('Comparaison des modèles — RMSE (plus bas = meilleur)')
ax.set_xlabel('RMSE')
ax.legend()

plt.tight_layout()
plt.savefig('../outputs/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

### 📋 Tableau comparatif des modèles

| Modèle | RMSE | vs Baseline |
|--------|------|-------------|
| Linear Regression | 0.4116 | référence |
| Ridge | 0.4136 | +0.49% ❌ |
| Lasso | 0.4146 | +0.73% ❌ |
| Random Forest | 0.4010 | -2.57% ✅ |
| Gradient Boosting | 0.3977 | -3.37% ✅ |
| XGBoost | 0.3977 | -3.37% ✅ |
| **LightGBM** | **0.3881** | **-5.70% ✅ 🏆** |

**Meilleur modèle : LightGBM avec un RMSE de 0.3881**

**Observations :**
- Les modèles linéaires (Ridge, Lasso) ne battent même pas la baseline — 
  ils ne capturent pas les relations non-linéaires entre les features
- Les modèles à base d'arbres (Random Forest, Gradient Boosting, XGBoost, LightGBM) 
  battent tous la baseline — ils gèrent mieux la complexité des données
- LightGBM est le grand gagnant avec 5.70% d'amélioration sur la baseline
- XGBoost et Gradient Boosting sont ex-aequo à 0.3977

**→ On va optimiser LightGBM dans l'étape suivante pour essayer 
de descendre encore plus bas !**

### Étape 13 — Optimisation du meilleur modèle (LightGBM)

LightGBM est notre meilleur modèle avec RMSE = 0.3881.
On va maintenant l'optimiser avec un **RandomizedSearchCV** pour 
trouver les meilleurs hyperparamètres et améliorer encore ce score.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

# Grille d'hyperparamètres à tester
param_grid = {
    'n_estimators'  : [100, 200, 300, 500],
    'max_depth'     : [3, 5, 7, 10, -1],
    'learning_rate' : [0.01, 0.05, 0.1, 0.2],
    'subsample'     : [0.6, 0.7, 0.8, 1.0],
    'num_leaves'    : [31, 50, 70, 100],
    'min_child_samples': [10, 20, 30]
}

lgbm_search = RandomizedSearchCV(
    LGBMRegressor(random_state=42, verbose=-1, n_jobs=-1),
    param_distributions=param_grid,
    n_iter=30,          # teste 30 combinaisons aléatoires
    cv=5,               # cross-validation 5 folds
    scoring='neg_root_mean_squared_error',
    random_state=42,
    n_jobs=-1,
    verbose=1
)

print("Recherche des meilleurs hyperparamètres en cours...")
print("(peut prendre 2-3 minutes)\n")
lgbm_search.fit(X_train, y_train)

print(f"\nMeilleurs paramètres : {lgbm_search.best_params_}")
print(f"Meilleur RMSE CV    : {-lgbm_search.best_score_:.4f}")

In [ ]:
# Évaluation du modèle optimisé — score CV uniquement pour cohérence
rmse_best_cv = -lgbm_search.best_score_

print(f"RMSE LightGBM baseline  (CV) : 0.3881")
print(f"RMSE LightGBM optimisé  (CV) : {rmse_best_cv:.4f}")
print(f"Amélioration                 : {((0.3881 - rmse_best_cv) / 0.3881 * 100):.2f}%")

# Ajout aux résultats — tous en CV pour cohérence
results['LightGBM optimisé'] = rmse_best_cv

# On garde best_lgbm pour les étapes suivantes
best_lgbm = lgbm_search.best_estimator_
y_pred_best = best_lgbm.predict(X_val)
residuals_available = True

In [ ]:
# Les 20 features les plus importantes
feature_importance = pd.DataFrame({
    'feature'   : X.columns,
    'importance': best_lgbm.feature_importances_
}).sort_values('importance', ascending=False).head(20)

fig, ax = plt.subplots(figsize=(12, 8))
bars = ax.barh(feature_importance['feature'], 
               feature_importance['importance'],
               color='steelblue', alpha=0.8)
ax.set_title('Top 20 features les plus importantes — LightGBM')
ax.set_xlabel('Importance')
ax.invert_yaxis()

plt.tight_layout()
plt.savefig('../outputs/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Résidus = différence entre valeur réelle et prédiction
residuals = y_val - y_pred_best

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution des résidus
axes[0].hist(residuals, bins=50, color='steelblue', edgecolor='white')
axes[0].axvline(x=0, color='red', linestyle='--', linewidth=1.5)
axes[0].set_title('Distribution des résidus')
axes[0].set_xlabel('Résidu (réel - prédit)')
axes[0].set_ylabel('Nombre de logements')

# Résidus vs valeurs prédites
axes[1].scatter(y_pred_best, residuals, alpha=0.2, color='steelblue', s=5)
axes[1].axhline(y=0, color='red', linestyle='--', linewidth=1.5)
axes[1].set_title('Résidus vs valeurs prédites')
axes[1].set_xlabel('log_price prédit')
axes[1].set_ylabel('Résidu')

plt.tight_layout()
plt.savefig('../outputs/residuals_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nRésidu moyen    : {residuals.mean():.4f}")
print(f"Résidu std      : {residuals.std():.4f}")
print(f"Résidu max      : {residuals.max():.4f}")
print(f"Résidu min      : {residuals.min():.4f}")

### 📋 Observations optimisation

**Meilleurs hyperparamètres trouvés :**
- n_estimators : 200
- max_depth : 5
- learning_rate : 0.1
- subsample : 1.0
- num_leaves : 50
- min_child_samples : 10

**Résultats — tous en cross-validation 5 folds pour cohérence :**
- LightGBM baseline : RMSE CV = 0.3881
- LightGBM optimisé : RMSE CV = 0.3877
- Amélioration de 0.04% : le modèle de base était déjà bien calibré

**Feature importance — Top 5 :**
- `longitude` : la localisation GPS capture des effets de ville très précis
- `dist_center` : confirme l'importance de la proximité au centre-ville
- `neighbourhood_mean_price` : le target encoding du quartier est très informatif
- `host_seniority_days` : l'ancienneté de l'hôte est informative
- `latitude` : complète la localisation GPS

**Analyse des résidus :**
- Distribution centrée autour de 0 — pas de biais systématique observé
- La majorité des erreurs sont comprises entre -1 et +1
- Les erreurs les plus grandes concernent les logements aux prix extrêmes
- Le scatter résidus vs prédits montre une légère hétéroscédasticité 
  sur les prix bas

### Étape 14 — Entraînement final & génération des prédictions

On ré-entraîne le meilleur modèle (LightGBM optimisé) sur **tout** 
le train set (et non plus 80%) pour maximiser les informations 
disponibles, puis on prédit sur le test set.

In [ ]:
# Modèle final avec les meilleurs hyperparamètres trouvés
final_model = LGBMRegressor(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.1,
    subsample=1.0,
    num_leaves=50,
    min_child_samples=10,
    random_state=42,
    verbose=-1,
    n_jobs=-1
)

# Entraînement sur TOUT le train set cette fois
final_model.fit(X, y)

print("Modèle final entraîné sur tout le train set")
print(f"Nombre de lignes utilisées : {X.shape[0]}")
print(f"Nombre de features         : {X.shape[1]}")

In [ ]:
# Colonnes à supprimer du test set — mêmes que pour le train
cols_to_drop_test = [c for c in ['id', 'description', 'name', 
                                  'amenities', 'zipcode'] 
                     if c in test_fe.columns]

X_test = test_fe.drop(columns=cols_to_drop_test)

# Vérification que train et test ont les mêmes colonnes
print(f"Colonnes X_train : {X.shape[1]}")
print(f"Colonnes X_test  : {X_test.shape[1]}")

# Aligner au cas où
X_test = X_test.reindex(columns=X.columns, fill_value=0)
print(f"Colonnes X_test après alignement : {X_test.shape[1]}")

In [ ]:
# Prédictions sur le test set
predictions = final_model.predict(X_test)

print(f"Prédictions générées")
print(f"Nombre de prédictions : {len(predictions)}")
print(f"Min  : {predictions.min():.3f}")
print(f"Max  : {predictions.max():.3f}")
print(f"Mean : {predictions.mean():.3f}")

In [ ]:
# Génération du CSV au format attendu
# Récupération des ids depuis le fichier test original
test_ids = pd.read_csv('../data/raw/airbnb_test.csv', index_col=0).index

submission = pd.DataFrame({
    'logpred': predictions
}, index=test_ids)

# Vérification du format
print("Aperçu du fichier de prédiction :")
print(submission.head(5))
print(f"\nShape : {submission.shape}")

# Sauvegarde
submission.to_csv('../outputs/predictions.csv')
print("\nFichier predictions.csv sauvegardé")

## Conclusion — Analyse finale

### Résumé des méthodes testées

**Phase 1 — Exploration**
On a chargé et analysé 22 234 logements Airbnb répartis sur 6 villes 
américaines (NYC, LA, SF, DC, Chicago, Boston). L'exploration a révélé 
des valeurs manquantes sur plusieurs colonnes clés (host_response_rate 
24.6%, review_scores_rating 22.4%) ainsi que des variables textuelles 
nécessitant un traitement spécifique.

**Phase 2 — Feature Engineering**
On est passés de 28 colonnes brutes à 62 features numériques grâce à :
- Encodage OHE des variables catégorielles (room_type, city, bed_type, property_type)
- Parsing des amenities → 13 features binaires + n_amenities
- Features temporelles → host_seniority_days, review_span_days, has_no_reviews
- Features géographiques → dist_center (distance au centre-ville)
- Ratios → accommodates_per_bed, accommodates_per_bedroom
- Target encoding du neighbourhood → neighbourhood_mean_price

**Phase 3 — Modélisation**
7 modèles testés avec cross-validation 5 folds :

| Modèle | RMSE |
|--------|------|
| Linear Regression | 0.4116 |
| Ridge | 0.4136 |
| Lasso | 0.4146 |
| Random Forest | 0.4010 |
| Gradient Boosting | 0.3977 |
| XGBoost | 0.3977 |
| **LightGBM** | **0.3881** |

### Variables les plus importantes

Le modèle final LightGBM a identifié comme features les plus 
prédictives :
1. `longitude` — la localisation GPS capture des effets de ville très précis
2. `dist_center` — plus on est proche du centre, plus c'est cher
3. `neighbourhood_mean_price` — le target encoding du quartier est très informatif
4. `host_seniority_days` — l'ancienneté de l'hôte influence le prix
5. `n_amenities` — le standing du logement capturé par ses équipements

### Pourquoi LightGBM fonctionne mieux

Les modèles linéaires (Ridge, Lasso) ne capturent pas les relations 
non-linéaires entre les features — par exemple l'interaction entre 
`room_type` et `bedrooms`. LightGBM, basé sur des arbres de décision 
boostés, capture naturellement ces interactions complexes.

### Pistes d'amélioration

- **Stacking de modèles** : combiner LightGBM + XGBoost + Random Forest 
  pour obtenir un meilleur RMSE
- **Features NLP** : extraire des informations de la colonne `description` 
  avec du text mining (sentiment, mots clés comme "luxury", "cozy"...)
- **Features météo** : ajouter la saisonnalité selon les dates
- **Target encoding plus avancé** : utiliser le zipcode en plus du 
  neighbourhood pour affiner la localisation
- **Outlier detection** : identifier et traiter séparément les logements 
  de luxe (prix > e^6.5) qui sont difficiles à prédire

Ce sont des améliorations cruciales pour descendre sous le RMSE trouvé de 0.38.